In [25]:
!pip install opendatasets

In [26]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews


100%|██████████| 25.7M/25.7M [00:00<00:00, 1.33GB/s]

In [23]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
import torch.nn.functional as F
from nltk.corpus import stopwords
from collections import Counter
import string
import re
import seaborn as sns
from nltk.tokenize import word_tokenize
from tqdm import tqdm
from nltk.stem import PorterStemmer
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,TensorDataset
import torch.optim as optim

In [24]:
is_cuda = torch.cuda.is_available()

# If we have a GPU available, we'll set our device to GPU. We'll use this device variable later in our code.
if is_cuda:
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU not available, CPU used")

GPU is available


In [28]:
df = pd.read_csv("/content/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [29]:
df=df.sample(frac =.10)

In [30]:
df.head(2)

,review,sentiment
8328,"Now don't get me wrong, i love a good film and...",negative
27201,"Like a lot of series pilots, Dark Angel's open...",positive


Data Preprocessing

In [38]:
#Lower Case
df["review"]=df["review"].str.lower()

In [41]:
# REMOVE URL's.
import re
def remove_urls(text):
    return re.sub(r'http\S+', '', text)

In [43]:
df["review"] = df["review"].astype(str).apply(remove_urls)


In [44]:
#REMOVE PUNCTUATIONS AND EMOJI
import re

def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

In [45]:
df["review"] = df["review"].apply(remove_punctuations)

In [46]:
#REMOVE HTML
import re

def remove_html(text):
    text=re.sub(r'<.*?>', '', text)
    return text

In [47]:
df["review"] = df["review"].apply(remove_html)

In [66]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [62]:
#REMOVE STOPWORDS

def remove_stopword(text):
    stop_words = stopwords.words('english')  # Specify 'english' for English stopwords
    temp_text = word_tokenize(text)

    for word in temp_text:
        if word in stop_words:
            text=text.replace(word,"")
    return text

In [67]:
df["review"] = df["review"].apply(remove_stopword)

In [68]:
def Stemming(text):
    ps = PorterStemmer()
    tokens = word_tokenize(text)
    stemmed_words = []
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return ' '.join(stemmed_words)

In [69]:
df["review"] = df["review"].apply(Stemming)

In [70]:
df.head(3)

,review,sentiment
8328,nan,negative
27201,nan,positive
48477,nan,negative


Changing the Target values to categorical valu

In [71]:
df["sentiment"].replace("positive",0,inplace=True)
df["sentiment"].replace("negative",1,inplace=True)

In [72]:
Y=df["sentiment"]

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer()
X =tf.fit_transform(df['review']).toarray()

Split the dataset

In [74]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.20,random_state=0)

In [75]:
X_train.shape

(4000, 1)

In [76]:
shape=X_train.shape

In [77]:
shape[1]

1

In [78]:
X_test.shape

(1000, 1)

In [79]:
type(X_train)

numpy.ndarray

In [80]:
type(Y_train)

pandas.core.series.Series

In [81]:
Y_train = Y_train.to_numpy()
Y_test = Y_test.to_numpy()

In [82]:
X_train.ndim

2

Create Tensor Datasets

In [83]:
train_set = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(Y_train).float())
test_set = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(Y_test).float())

Data Loader (Load Data in Batches)

In [84]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [85]:
class Rnn(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully Connected Layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # RNN forward pass
        out, _ = self.rnn(x, h0)

        # Pass through fully connected layer
        out = self.fc(out[:, -1, :])
        return out

Hyperparameters

In [86]:
input_dim = shape[1] # Updated to match TF-IDF feature size
hidden_dim = 128
output_dim = 1  # Binary classification (positive or negative sentiment)
num_layers = 1
num_epochs = 10
batch_size = 64
learning_rate = 0.001

Initialize model, criterion, and optimizer

In [87]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Rnn(input_dim, hidden_dim, output_dim, num_layers).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Training

In [88]:
for epoch in range(num_epochs):
    model.train()
    for X_batch, Y_batch in train_loader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        # Add an additional dimension for the sequence length
        X_batch = X_batch.unsqueeze(1)

        outputs = model(X_batch)

        # Apply sigmoid activation to get probabilities
        outputs = torch.sigmoid(outputs.squeeze())

        # Compute the loss
        loss = criterion(outputs, Y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/10], Loss: 0.6959
Epoch [2/10], Loss: 0.6894
Epoch [3/10], Loss: 0.6922
Epoch [4/10], Loss: 0.6905
Epoch [5/10], Loss: 0.6925
Epoch [6/10], Loss: 0.6931
Epoch [7/10], Loss: 0.6929
Epoch [8/10], Loss: 0.6932
Epoch [9/10], Loss: 0.6932
Epoch [10/10], Loss: 0.6935


EVALUATION

In [89]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for X_batch, Y_batch in test_loader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        # Add an additional dimension for the sequence length
        X_batch = X_batch.unsqueeze(1)

        outputs = model(X_batch)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        total += Y_batch.size(0)
        correct += (predicted == Y_batch).sum().item()

    accuracy = correct / total
    print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 50.70%
